In [ ]:
import glob, os, numpy as np, pandas as pd
from scipy.stats import mannwhitneyu
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 400)
f = glob.glob('/kaggle/input/**/per_prototype_shift.csv', recursive=True); print(f)
D = os.path.dirname(f[0])
R = pd.read_csv(f'{D}/per_run.csv'); PR = pd.read_csv(f'{D}/per_prototype_shift.csv')
FAM = {'PPN-indep': 'PPN', 'Correia-indep': 'PPN-FA-Correia', 'Ours-indep': 'Ours'}
R['fam'] = R.config.replace(FAM); PR['fam'] = PR.config.replace(FAM)
cols = ['run', 'auc', 'auc_keep', 'auc_bg', 'auc_lo'] + [c for c in R.columns if c.startswith('flip ') or c.startswith('dmax ')] + ['lesion_protos', 'aligned', 'mean_best_auc_crit']
print(R[[c for c in cols if c in R.columns]].round(4).to_string(index=False))

# detector-free evidence shift: d of melanoma prototypes (j >= 10) minus d of benign prototypes, weighted by |own-class weight|
PR['cls'] = (PR.proto >= 10).astype(int)
def shift(g):
    m, b = g[g.cls == 1], g[g.cls == 0]
    return pd.Series(dict(mel=np.average(m.d, weights=m.w), ben=np.average(b.d, weights=b.w), dmax=g.d.max(), dmin=g.d.min(),
                          n025=int((g.d > 0.25).sum()), q90=g.d.quantile(0.9), absmean=g.d.abs().mean()))
S = PR.groupby(['run', 'fam', 'kind']).apply(shift).reset_index(); S['mel_minus_ben'] = S.mel - S.ben
print('\nper run and insertion (d = mean activation change / between-image SD):')
print(S.round(3).to_string(index=False))
print('\nmean over runs by family:')
print(S.groupby(['kind', 'fam'])[['mel', 'ben', 'mel_minus_ben', 'dmax', 'dmin', 'n025', 'q90', 'absmean']].mean().round(3).to_string())
POOL = ['PPN-std', 'PPN', 'PPN-FA']
rows = []
for kind in S.kind.unique():
    for met in ['mel_minus_ben', 'dmax', 'absmean']:
        a = S[(S.kind == kind) & (S.fam == 'Ours')][met].values
        for other in ['PPN', 'PPN-FA-Correia', 'PPN-std', 'PPN-FA', 'Ours-noLes', 'Ours-stdPush', 'POOL13']:
            b = S[(S.kind == kind) & (S.fam.isin(POOL) if other == 'POOL13' else S.fam == other)][met].values
            meth = 'exact' if len(set(a) | set(b)) == len(a) + len(b) else 'asymptotic'
            rows.append(dict(kind=kind, metric=met, versus=other, ours=a.mean(), other=b.mean(), n=(len(a), len(b)),
                             p=mannwhitneyu(a, b, alternative='two-sided', method=meth).pvalue))
print('\nOurs (7 runs) versus others across runs, exact Mann-Whitney:')
print(pd.DataFrame(rows).round(4).to_string(index=False))
# relation between evidence shift and benign->melanoma flips across runs
fl = R.melt(id_vars=['run'], value_vars=[c for c in R.columns if c.startswith('flip ')], var_name='kind', value_name='flip')
fl['kind'] = fl.kind.str.replace('flip ', '')
M = S.merge(fl, on=['run', 'kind'])
from scipy.stats import spearmanr
for kind, g in M.groupby('kind'):
    print(kind, 'Spearman(mel_minus_ben, flip) over', len(g), 'runs:', np.round(spearmanr(g.mel_minus_ben, g.flip), 4))
print('NB11b FINISHED')